In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    try:
        %reload_ext rpy2.ipython
    except Exception as e2:
        print("Note on rpy2 initialization:", e2)

# 3-Tier Master Triage Pipeline Benchmark (`models/combined_hierarchical_triage_pipeline.ipynb`)

This notebook evaluates the **3-Tier 4-Model Hierarchical LightGBM Triage Pipeline** comparing two routing algorithms on the **1% Holdout Test Set**:

### 3-Tier 4-Model Hierarchical Architecture
- **Layer 1 LightGBM (ESI 1 Detector)** (`lightgbm_layer1_esi1_model.rds`): Binary Classifier trained on 5 Clinical Binary Features.
- **Layer 2 LightGBM (ESI 2/3 vs 4/5 Specialist)** (`rf_esi23_esi45_extreme_model.rds`): Binary Classifier trained on non-ESI 1 data with 35 Standard Features.
- **Layer 3A LightGBM (ESI 2 vs ESI 3 Specialist)** (`lightgbm_esi23_model.rds`): Binary Classifier trained on ESI 2 & 3 rows with 35 Standard Features.
- **Layer 3B LightGBM (ESI 4 vs ESI 5 Specialist)** (`lightgbm_esi45_model.rds`): Binary Classifier trained on ESI 4 & 5 rows with 4 Clinical Binary Features.

### Evaluated Routing Algorithms
1. **Probabilistic Joint Product Algorithm (Soft Scaling)**:
   - $P(\text{ESI 1}) = P_{L1}(\text{ESI 1})$
   - $P(\text{ESI 2}) = (1 - P_{L1}) \times P_{L2}(\text{ESI 2/3}) \times P_{L3A}(\text{ESI 2})$
   - $P(\text{ESI 3}) = (1 - P_{L1}) \times P_{L2}(\text{ESI 2/3}) \times P_{L3A}(\text{ESI 3})$
   - $P(\text{ESI 4}) = (1 - P_{L1}) \times (1 - P_{L2}) \times P_{L3B}(\text{ESI 4})$
   - $P(\text{ESI 5}) = (1 - P_{L1}) \times (1 - P_{L2}) \times P_{L3B}(\text{ESI 5})$
2. **Hard Case-Selector Routing Algorithm (If-Else Routing)**:
   - `if L1 >= l1_threshold -> ESI 1; else if L2 >= 0.5 -> (if L3A >= 0.5 -> ESI 2 else ESI 3); else -> (if L3B >= 0.5 -> ESI 4 else ESI 5)`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
# Unique Column Dataframe
df_master <- data.frame(
  age = raw_df$age, gender = gender_vec, cc_breathingdifficulty = cc_bd_vec,
  triage_vital_hr = t_hr, triage_vital_sbp = t_sbp, triage_vital_rr = t_rr, triage_vital_o2 = t_o2,
  pulse_last = pulse_last, resp_last = resp_last, spo2_last = spo2_last, sbp_last = sbp_last,
  pulse_min = pulse_min, resp_min = resp_min, spo2_min = spo2_min, sbp_min = sbp_min,
  pulse_max = pulse_max, resp_max = resp_max, spo2_max = spo2_max, sbp_max = sbp_max,
  hr_mean_to_last = t_hr - pulse_last, sbp_mean_to_last = t_sbp - sbp_last, spo2_mean_to_last = t_o2 - spo2_last, rr_mean_to_last = t_rr - resp_last,
  hr_range = pulse_max - pulse_min, rr_range = resp_max - resp_min, spo2_range = spo2_max - spo2_min, sbp_range = sbp_max - sbp_min,
  hr_last_to_min = pulse_last - pulse_min, rr_last_to_min = resp_last - resp_min, spo2_last_to_min = spo2_last - spo2_min, sbp_last_to_min = sbp_last - sbp_min,
  hr_last_to_max = pulse_last - pulse_max, rr_last_to_max = resp_last - resp_max, spo2_last_to_max = spo2_last - spo2_max, sbp_last_to_max = sbp_last - sbp_max,
  is_dyspnea_moderate     = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_tachypnea             = ifelse(t_rr > 30, 1, 0),
  is_bradypnea             = ifelse(t_rr < 10, 1, 0),
  is_tachycardia_total     = ifelse(t_hr > 150, 1, 0),
  is_hypotension             = ifelse(t_sbp <= 90, 1, 0),
  is_bradycardia_total       = ifelse(t_hr < 40, 1, 0),
  is_bradycardia_moderate    = ifelse(t_hr > 40 & t_hr < 60, 1, 0)
)
std_feat_names <- c(
  "age", "gender", "cc_breathingdifficulty", "triage_vital_hr", "triage_vital_sbp", "triage_vital_rr", "triage_vital_o2",
  "pulse_last", "resp_last", "spo2_last", "sbp_last", "pulse_min", "resp_min", "spo2_min", "sbp_min",
  "pulse_max", "resp_max", "spo2_max", "sbp_max", "hr_mean_to_last", "sbp_mean_to_last", "spo2_mean_to_last", "rr_mean_to_last",
  "hr_range", "rr_range", "spo2_range", "sbp_range", "hr_last_to_min", "rr_last_to_min", "spo2_last_to_min", "sbp_last_to_min",
  "hr_last_to_max", "rr_last_to_max", "spo2_last_to_max", "sbp_last_to_max"
)
l1_feat_names  <- c("is_dyspnea_moderate", "is_hypertension", "is_tachypnea", "is_bradypnea", "is_tachycardia_total")
l3b_feat_names <- c("is_hypotension", "is_bradycardia_total", "is_dyspnea_moderate", "is_bradycardia_moderate")
raw_esi <- as.character(raw_df[[target_col_name]])
df_master$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_master <- na.omit(df_master)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_master$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_master[in_train_val, ]
test_df      <- df_master[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
binary_cols <- c("gender", "cc_breathingdifficulty")
cont_cols   <- setdiff(std_feat_names, binary_cols)
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
val_scaled  <- val_df
val_scaled[, cont_cols]  <- predict(preproc, val_df[, cont_cols])
test_scaled <- test_df
test_scaled[, cont_cols] <- predict(preproc, test_df[, cont_cols])
cat(sprintf("Holdout Test Data Partition Summary: %d samples\n", nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load 4 Sub-Model Artifacts from deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
lgb_l1_obj  <- readRDS(file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
lgb_l2_obj  <- readRDS(file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
lgb_l3a_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi23_model.rds"))
lgb_l3b_obj <- readRDS(file.path(deploy_dir, "lightgbm_esi45_model.rds"))
l1_model  <- if (is.list(lgb_l1_obj)  && "model" %in% names(lgb_l1_obj))  lgb_l1_obj$model  else lgb_l1_obj
l2_model  <- if (is.list(lgb_l2_obj)  && "model" %in% names(lgb_l2_obj))  lgb_l2_obj$model  else lgb_l2_obj
l3a_model <- if (is.list(lgb_l3a_obj) && "model" %in% names(lgb_l3a_obj)) lgb_l3a_obj$model else lgb_l3a_obj
l3b_model <- if (is.list(lgb_l3b_obj) && "model" %in% names(lgb_l3b_obj)) lgb_l3b_obj$model else lgb_l3b_obj
cat("All 4 Sub-Models (Layer 1, Layer 2, Layer 3A, Layer 3B) successfully loaded!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Run 3-Tier Inference & Compute Joint Probabilities
# ---------------------------------------------------------
X_val_l1   <- as.matrix(val_scaled[, l1_feat_names])
X_test_l1  <- as.matrix(test_scaled[, l1_feat_names])
X_test_std <- as.matrix(test_scaled[, std_feat_names])
X_test_l3b <- as.matrix(test_scaled[, l3b_feat_names])
# Calibrate Layer 1 threshold on validation split via 99th percentile
p1_val <- predict(l1_model, X_val_l1)
l1_threshold <- quantile(p1_val, 0.99, na.rm = TRUE)
# Sub-Model Probabilities
p1  <- predict(l1_model,  X_test_l1)  # P(ESI 1)
p2  <- predict(l2_model,  X_test_std) # P(ESI 2/3)
p3a <- predict(l3a_model, X_test_std) # P(ESI 2 | ESI 2/3)
p3b <- predict(l3b_model, X_test_l3b) # P(ESI 4 | ESI 4/5)
# ---------------------------------------------------------
# 1. Soft Probabilistic Joint Product Scaling
# ---------------------------------------------------------
probs_soft <- matrix(0, nrow = nrow(test_df), ncol = 5)
colnames(probs_soft) <- c("1", "2", "3", "4", "5")
probs_soft[, 1] <- p1
probs_soft[, 2] <- (1 - p1) * p2 * p3a
probs_soft[, 3] <- (1 - p1) * p2 * (1 - p3a)
probs_soft[, 4] <- (1 - p1) * (1 - p2) * p3b
probs_soft[, 5] <- (1 - p1) * (1 - p2) * (1 - p3b)
preds_soft <- factor(apply(probs_soft, 1, which.max), levels = 1:5)
# ---------------------------------------------------------
# 2. Hard Case-Selector Routing (If-Else with Calibrated L1 Threshold)
# ---------------------------------------------------------
preds_hard <- numeric(nrow(test_df))
for (i in 1:nrow(test_df)) {
  if (p1[i] >= l1_threshold) {
    preds_hard[i] <- 1
  } else if (p2[i] >= 0.5) {
    preds_hard[i] <- ifelse(p3a[i] >= 0.5, 2, 3)
  } else {
    preds_hard[i] <- ifelse(p3b[i] >= 0.5, 4, 5)
  }
}
preds_hard <- factor(preds_hard, levels = 1:5)
act_test <- factor(as.numeric(as.character(test_df$target_col)), levels = 1:5)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Benchmark Metrics Comparison (Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
eval_algorithm <- function(preds, probs, alg_name) {
  cm <- confusionMatrix(preds, act_test)
  
  rec_list  <- as.numeric(cm$byClass[, "Sensitivity"])
  spec_list <- as.numeric(cm$byClass[, "Specificity"])
  bal_list  <- as.numeric(cm$byClass[, "Balanced Accuracy"])
  
  rec_list[is.na(rec_list)]   <- 0
  spec_list[is.na(spec_list)] <- 0
  bal_list[is.na(bal_list)]   <- 0
  
  auc_list <- sapply(1:5, function(i) {
    act_bin <- ifelse(act_test == i, 1, 0)
    if (!is.null(probs)) {
      p_col <- probs[, i]
    } else {
      p_col <- ifelse(preds == i, 1, 0)
    }
    r_obj <- tryCatch(pROC::roc(act_bin, p_col), error = function(e) NULL)
    if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  })
  
  macro_rec  <- mean(rec_list)
  macro_spec <- mean(spec_list)
  macro_bal  <- mean(bal_list)
  macro_auc  <- mean(auc_list, na.rm = TRUE)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   5-CLASS BENCHMARK: %s\n", toupper(alg_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Macro Recall (Sens)     : %.4f\n", macro_rec))
  cat(sprintf("  Macro Specificity       : %.4f\n", macro_spec))
  cat(sprintf("  Macro Balanced Accuracy : %.4f\n", macro_bal))
  cat(sprintf("  Macro ROC-AUC           : %.4f\n", macro_auc))
  cat(sprintf("============================================================\n\n"))
  cat("Confusion Matrix (Reference = Ground Truth, Prediction = Model):\n")
  print(cm$table)
  cat("\n\n")
  
  return(data.frame(
    Algorithm = alg_name,
    Macro_Recall = round(macro_rec, 4),
    Macro_Specificity = round(macro_spec, 4),
    Macro_Balanced_Accuracy = round(macro_bal, 4),
    Macro_ROC_AUC = round(macro_auc, 4)
  ))
}
soft_res <- eval_algorithm(preds_soft, probs_soft, "Probabilistic_Joint_Product")
hard_res <- eval_algorithm(preds_hard, NULL,       "Hard_Case_Selector_IfElse")
comp_df <- rbind(soft_res, hard_res)
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
write.csv(comp_df, file = file.path(reports_dir, "combined_pipeline_test_report.csv"), row.names = FALSE)
cat("Benchmark Comparison Report written to reports/combined_pipeline_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Grouped Bar Chart Visualization
# ---------------------------------------------------------
df_long <- comp_df %>%
  pivot_longer(cols = c("Macro_Recall", "Macro_Specificity", "Macro_Balanced_Accuracy", "Macro_ROC_AUC"),
               names_to = "Metric", values_to = "Score") %>%
  mutate(Metric = factor(Metric, levels = c("Macro_Recall", "Macro_Specificity", "Macro_Balanced_Accuracy", "Macro_ROC_AUC")))
p <- ggplot(df_long, aes(x = Metric, y = Score, fill = Algorithm)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7) +
  geom_text(aes(label = sprintf("%.4f", Score)), position = position_dodge(width = 0.8), vjust = -0.4, size = 3.5) +
  scale_y_continuous(limits = c(0, 1.05), breaks = seq(0, 1, 0.1)) +
  scale_fill_manual(values = c("Probabilistic_Joint_Product" = "#1f77b4", "Hard_Case_Selector_IfElse" = "#ff7f0e")) +
  labs(title = "Master 3-Tier Pipeline: Soft Probabilistic Joint vs Hard Case-Selector",
       subtitle = "Holdout Test Set Performance across Macro Metrics",
       x = "Metric", y = "Score", fill = "Algorithm") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top", plot.title = element_text(face = "bold", hjust = 0.5), plot.subtitle = element_text(hjust = 0.5))
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
ggsave(filename = file.path(plots_dir, "combined_pipeline_metrics_barchart.png"), plot = p, width = 9, height = 5.5, dpi = 300)
cat("Bar chart saved to plots/combined_pipeline_metrics_barchart.png\n")